# Margen ±: el error base

El margen de la app junta dos errores: `± = √(error base² + error por lo que falta²)`.

- **Error por lo que falta:** la mediana del cambio de la edad según cuántos valores faltan (`missing_values.ipynb`).
- **Error base** (este notebook): la calibración en años. En cada décimo de riesgo, cuántos años habría que mover la edad biológica para que el riesgo predicho cuadre con las muertes de verdad.

Cada persona se evalúa con un modelo entrenado sin ella (los 5 bloques de train); el test no se toca. Es ruidoso en riesgo bajo, porque casi nadie muere, así que se resume en dos tramos: **riesgo bajo** (décimos 1 a 5) y **el resto** (6 a 10). El error de cada tramo es la media del error, sin signo, de sus décimos.

In [1]:
# Imports
import numpy as np
import pandas as pd

from bioage.config.model_levels import LEVEL_COLUMNS
from bioage.config.paths import GOLD_TABLE
from bioage.modeling.s1_split import split_data
from bioage.modeling.s3_algorithms import build_pipeline, survival_target
from bioage.modeling.s4_biological_age import biological_age, fit_reference, risk_at_horizon
from bioage.modeling.s5_evaluation import calibration_in_years

C:\Data\1_MisCosas\Evolve\TFM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
LEVELS = ["routine", "extras", "questionnaire", "full"]
LOW_RISK = range(5)  # tenths of risk

gold = pd.read_parquet(GOLD_TABLE)
train, test, folds = split_data(gold)

## 1. Calibración en años, en cada bloque

In [3]:
tables, risks = {}, {}
for level in LEVELS:
    risk = pd.Series(index=train.index, dtype=float)
    for fold, (fit_rows, validation_rows) in enumerate(folds):
        fit, validation = train.iloc[fit_rows], train.iloc[validation_rows]
        pipeline = build_pipeline("cox", LEVEL_COLUMNS[level]).fit(fit, survival_target(fit))
        reference = fit_reference(risk_at_horizon(pipeline, fit), fit)
        risk.loc[validation.index] = risk_at_horizon(pipeline, validation)
        age = biological_age(risk.loc[validation.index], validation["sex"], reference)
        tables[(level, fold)] = calibration_in_years(risk.loc[validation.index], age, validation, reference)
        print(f"{level}: fold {fold + 1}", flush=True)
    risks[level] = risk

routine: fold 1


routine: fold 2


routine: fold 3


routine: fold 4


routine: fold 5


extras: fold 1


... (10 outputs left out)

full: fold 2


full: fold 3


full: fold 4


full: fold 5


In [4]:
years = pd.DataFrame({key: table["years"] for key, table in tables.items()})
years.index.name = "tenth of risk"
years.round(1)

routine                      extras                       \
                    0    1     2    3    4      0    1     2    3    4   
tenth of risk                                                            
0                -6.9  5.2   6.8  8.0 -0.6    2.8 -6.1  12.9 -5.0  1.0   
1               -12.8  6.0   7.7  3.7 -4.3    6.7  6.2   8.5  8.3 -6.1   
2                10.0  5.4  11.5  4.4 -1.5    1.7  8.9  12.2  7.6 -7.8   
3                -8.4 -0.7  -9.6  4.3 -4.4   -0.5 -5.8  -8.2  5.4  2.4   
4                -1.6 -2.4  -0.6  2.2 -2.3   -8.5 -3.4   0.8  1.5 -5.5   
5                -4.5  0.3  -0.2 -2.1  1.8   -3.1 -0.2  -7.6 -3.1 -0.2   
6                 2.4 -1.0   1.9 -2.5 -4.4   -1.9 -3.5  -0.4 -0.6 -2.7   
7                 0.4 -2.0  -0.1 -0.5  2.0   -0.3 -0.4   0.7 -2.4  0.4   
8                -0.4  0.7   1.3 -1.3  0.1    0.0  1.2   0.2  0.9  0.1   
9                -0.5  0.6   0.8 -0.7  0.2    0.1  0.9   1.6 -0.9  0.6   

              questionnaire                       full                        
                          0    1     2    3     4    0    1     2    3     4  
tenth of risk                                                                 
0                       7.1  5.9  16.0  9.1   3.1  9.2  2.3  15.4 -2.4  -2.7  
1                       5.3  3.1   5.3  9.7   2.6  3.1  9.0  13.8  8.2   0.1  
2                       2.2  6.2  10.9  4.5  -5.5  4.4  1.7   1.6  9.8  -1.5  
3                      -2.9  5.6  -7.0 -1.3 -11.0 -4.1  4.2   3.0 -2.9 -15.5  
4                      -2.4 -2.1   1.4  1.2   1.3 -2.5  0.5  -1.9  3.3  -1.5  
5                      -5.2 -8.1  -0.2 -0.8  -3.1 -5.3 -6.5  -1.2 -0.3   0.3  
6                       2.6 -2.5  -2.8 -3.5  -1.1 -3.8 -3.8   1.1 -3.7  -2.5  
7                      -1.8 -0.5   1.1 -1.3  -0.1  1.2 -0.6  -3.1 -2.1   1.1  
8                       0.2  1.3   1.0 -0.7   0.6 -1.7  1.5   1.4  0.4  -1.4  
9                      -0.4  0.2   1.1 -0.2   0.4  0.6  0.6   1.2 -0.3   1.2

## 2. Error base de cada tramo

In [5]:
rows = {}
for level in LEVELS:
    size = years[level].abs().mean(axis=1)
    rows[level] = {
        "low risk below": risks[level].median(),
        "low risk: years": size.loc[list(LOW_RISK)].mean(),
        "rest: years": size.drop(list(LOW_RISK)).mean(),
    }
base = pd.DataFrame(rows).T
base.round(4)

,low risk below,low risk: years,rest: years
routine,0.0366,5.2387,1.2993
extras,0.0331,5.7488,1.3559
questionnaire,0.0344,5.3157,1.6322
full,0.0317,4.9826,1.8770


## 3. Conclusión

**El error base en riesgo bajo es grande y muy ruidoso.** En los décimos 1 a 5 el error de cada bloque salta de −13 a +16 años: casi nadie muere a esas edades, y a esos riesgos una diferencia mínima son muchos años. Desde el décimo 6 se queda casi siempre por debajo de 3 años.

| Nivel | Riesgo bajo: por debajo de | Error base, riesgo bajo | Error base, el resto |
|---|---|---|---|
| `routine` | 3,7 % | 5,2 años | 1,3 años |
| `extras` | 3,3 % | 5,7 años | 1,4 años |
| `questionnaire` | 3,4 % | 5,3 años | 1,6 años |
| `full` | 3,2 % | 5,0 años | 1,9 años |

Van a `BASE_MARGIN`, en `config/app.py`.

**Consecuencia:** la mitad de la gente, la de riesgo bajo (sobre todo menores de unos 45 años), tendrá un ± de 5 años o más aunque traiga la analítica completa, así que verá el aviso gris de precaución.